# Solution — TP 2 : Auto-Encodeurs (AE) et Auto-Encodeurs Variationnels (VAE)
## Exercice 1 : Auto-Encodeur convolutif sur CIFAR-10

Ce notebook contient la **solution complète de l'exercice 1** du TP (les parties AE / VAE sur Fashion-MNIST sont des exemples fournis dans l'énoncé).

| Étape de l'énoncé | Section |
|---|---|
| 1. Charger CIFAR-10, normaliser, créer les DataLoaders | Étape 1 |
| 2. Construire l'auto-encodeur convolutif (code latent de 64 dimensions) | Étape 2 |
| 3. Entraîner (Adam, lr = 1e-3, MSE, 10 époques, perte affichée à chaque époque) | Étape 3 |
| 4. Tester la robustesse au bruit (`+ 0.3 * torch.randn_like(images)`) | Étape 4 |
| 6. Visualiser originales / bruitées / reconstruites et commenter | Étape 5 |
| Bonus : Denoising AE (entraîné avec du bruit) | Bonus |

**Conseil :** sur Google Colab, activez un GPU (*Exécution → Modifier le type d'exécution → GPU*). Sur CPU, l'entraînement fonctionne mais est plus lent.

## Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

# Reproductibilité
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Exécution sur : {device}")

# Hyperparamètres de l'énoncé
LATENT_DIM = 64      # taille du code latent
EPOCHS     = 10      # nombre d'époques
LR         = 1e-3    # taux d'apprentissage (Adam)
BATCH_SIZE = 128
NOISE_STD  = 0.3     # écart-type du bruit gaussien (étape 4)

## Étape 1 — Charger CIFAR-10, normaliser et créer les DataLoaders

CIFAR-10 contient **50 000 images d'entraînement** et **10 000 images de test** (32×32 pixels, 3 canaux RGB, 10 classes).

**Normalisation :** `ToTensor()` convertit chaque image en tenseur de forme `(3, 32, 32)` et **divise les pixels par 255**, ce qui donne des valeurs dans $[0, 1]$.
On garde volontairement cet intervalle (et non une standardisation moyenne/écart-type) car le décodeur se termine par une **Sigmoid**, dont la sortie est dans $[0, 1]$ : l'entrée et la sortie doivent être dans la même plage pour que la MSE ait un sens.

In [ ]:
train_dataset = CIFAR10(root="./data/CIFAR10", train=True,  download=True, transform=ToTensor())
test_dataset  = CIFAR10(root="./data/CIFAR10", train=False, download=True, transform=ToTensor())

# num_workers=0 : évite les erreurs de multiprocessing sous Windows / Jupyter
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Images d'entraînement :", len(train_dataset))
print("Images de test        :", len(test_dataset))
x0, y0 = train_dataset[0]
print("Forme d'une image     :", tuple(x0.shape), "| min =", float(x0.min()), "| max =", float(x0.max()))
print("Classes               :", train_dataset.classes)

# Aperçu : 16 images d'entraînement
imgs = torch.stack([train_dataset[i][0] for i in range(16)])
plt.figure(figsize=(8, 3))
plt.imshow(make_grid(imgs, nrow=8, padding=2).permute(1, 2, 0).numpy())
plt.axis("off"); plt.title("Échantillon d'images CIFAR-10"); plt.show()

## Étape 2 — Construire l'auto-encodeur convolutif

On utilise `Conv2d` / `ConvTranspose2d` avec des activations `ReLU` et une sortie `Sigmoid`.
Avec `kernel_size=4, stride=2, padding=1`, la formule de convolution donne :

$$H_{out} = \left\lfloor \frac{H_{in} - 4 + 2\times 1}{2} \right\rfloor + 1 = \frac{H_{in}}{2}$$

(la taille est donc **divisée par 2** à chaque convolution et **multipliée par 2** à chaque convolution transposée).

| Étape | Encodeur | Forme de sortie |
|---|---|---|
| Entrée | image | 3 × 32 × 32 |
| Conv 1 + ReLU | 3 → 32 canaux | 32 × 16 × 16 |
| Conv 2 + ReLU | 32 → 64 canaux | 64 × 8 × 8 |
| Conv 3 + ReLU | 64 → 128 canaux | 128 × 4 × 4 |
| Flatten + Linear | 2048 → 64 | **64 (code latent z)** |

Le décodeur fait le chemin inverse : `Linear` (64 → 2048), redimensionnement en 128 × 4 × 4, puis 3 `ConvTranspose2d` (4 → 8 → 16 → 32) et une `Sigmoid`.
La compression est de $3072 / 64 = 48$ (×48).

In [ ]:
class ConvAutoEncoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        # --- ENCODEUR : 3x32x32 -> latent_dim ---
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),    # -> 32x16x16
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),   # -> 64x8x8
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # -> 128x4x4
            nn.ReLU(),
            nn.Flatten(),                                            # -> 2048
            nn.Linear(128 * 4 * 4, latent_dim),                      # -> latent_dim
        )
        # --- DÉCODEUR : latent_dim -> 3x32x32 ---
        self.decoder_linear = nn.Linear(latent_dim, 128 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # -> 64x8x8
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # -> 32x16x16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),    # -> 3x32x32
            nn.Sigmoid(),                                                     # sortie dans [0, 1]
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        h = torch.relu(self.decoder_linear(z)).view(-1, 128, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z   # reconstruction et code latent


net = ConvAutoEncoder(LATENT_DIM).to(device)

# Vérification des formes avec un mini-batch
with torch.no_grad():
    xb = torch.randn(4, 3, 32, 32, device=device)
    rec, z = net(xb)
print("Entrée         :", tuple(xb.shape))
print("Code latent z  :", tuple(z.shape))
print("Reconstruction :", tuple(rec.shape))
n_params = sum(p.numel() for p in net.parameters())
print(f"Nombre de paramètres : {n_params:,}")
print(f"Compression : 3072 -> {LATENT_DIM} (x{3072 // LATENT_DIM})")

## Étape 3 — Entraîner le modèle

- Optimiseur : **Adam** (`lr = 1e-3`) ;
- Perte : **MSE** $\mathcal{L}(x, \hat{x}) = \frac{1}{N}\sum (x - \hat{x})^2$ ;
- **10 époques**, avec affichage de la perte moyenne d'entraînement **et de test** à chaque époque.

La fonction accepte un paramètre `noise_std` : à `0` (par défaut) c'est un AE classique ; si `noise_std > 0`, on bruite l'entrée mais on garde l'image propre comme cible (c'est le **Denoising AE** du bonus).

In [ ]:
def train(net, train_loader, test_loader, epochs=EPOCHS, lr=LR, noise_std=0.0, name="AE"):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {"train": [], "test": []}

    for epoch in range(1, epochs + 1):
        # ---- Entraînement ----
        net.train()
        total, n = 0.0, 0
        for images, _ in train_loader:
            images = images.to(device)
            inputs = images
            if noise_std > 0:   # Denoising AE : entrée bruitée, cible propre
                inputs = torch.clamp(images + noise_std * torch.randn_like(images), 0, 1)
            recon, _ = net(inputs)
            loss = criterion(recon, images)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = total / n

        # ---- Évaluation (sur le jeu de test) ----
        net.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                inputs = images
                if noise_std > 0:
                    inputs = torch.clamp(images + noise_std * torch.randn_like(images), 0, 1)
                recon, _ = net(inputs)
                total += criterion(recon, images).item() * images.size(0)
                n += images.size(0)
        test_loss = total / n

        history["train"].append(train_loss)
        history["test"].append(test_loss)
        print(f"[{name}] Époque {epoch:2d}/{epochs} | perte train = {train_loss:.5f} | perte test = {test_loss:.5f}")
    return history


hist_ae = train(net, train_loader, test_loader, epochs=EPOCHS, name="AE")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), hist_ae["train"], "o-", label="Entraînement")
plt.plot(range(1, EPOCHS + 1), hist_ae["test"],  "s--", label="Test")
plt.xlabel("Époque"); plt.ylabel("MSE moyenne"); plt.yscale("log")
plt.title("Courbe d'apprentissage de l'AE"); plt.grid(alpha=0.3); plt.legend(); plt.show()

**Lecture de la courbe :** la perte diminue rapidement lors des premières époques puis se stabilise. Si les pertes d'entraînement et de test restent proches, il n'y a **pas de surapprentissage** : le modèle est plutôt limité par son goulot d'étranglement (64 valeurs pour 3072) que par un manque de données.

## Étape 4 — Tester la robustesse au bruit

On ajoute un bruit gaussien aux images de test : $x_{bruit} = \text{clip}(x + 0.3\,\varepsilon,\ 0,\ 1)$ avec $\varepsilon \sim \mathcal{N}(0, I)$, puis on passe ces images bruitées dans le modèle **déjà entraîné** (sans réentraînement).

Pour **quantifier** la qualité, on calcule sur tout le jeu de test :
- la **MSE** par rapport à l'image **propre** ;
- le **PSNR** (en dB, plus il est élevé mieux c'est) : $\text{PSNR} = 10\log_{10}\left(\frac{1}{\text{MSE}}\right)$.

In [ ]:
def add_noise(images, std=NOISE_STD, generator=None):
    noise = torch.randn(images.shape, generator=generator, device=images.device)
    return torch.clamp(images + std * noise, 0, 1)

@torch.no_grad()
def evaluate(net, loader, mode="clean", std=NOISE_STD, seed=123):
    """
    mode = "clean"   : AE appliqué aux images propres
    mode = "noisy"   : AE appliqué aux images bruitées (cible = image propre)
    mode = "noisy_input" : pas de modèle, on compare directement image bruitée / image propre
    Renvoie (MSE moyenne, PSNR moyen en dB).
    """
    net.eval()
    g = torch.Generator(device=device); g.manual_seed(seed)   # même bruit à chaque appel
    mse_sum, psnr_sum, n = 0.0, 0.0, 0
    for images, _ in loader:
        images = images.to(device)
        noisy = add_noise(images, std, g)
        if mode == "clean":
            out = net(images)[0]
        elif mode == "noisy":
            out = net(noisy)[0]
        else:
            out = noisy
        mse_img = ((out - images) ** 2).flatten(1).mean(1)           # MSE de chaque image
        psnr_img = 10 * torch.log10(1.0 / mse_img.clamp_min(1e-10))  # PSNR de chaque image
        mse_sum += mse_img.sum().item(); psnr_sum += psnr_img.sum().item(); n += images.size(0)
    return mse_sum / n, psnr_sum / n

rows = []
for label, mode in [("Image bruitée (sans modèle)", "noisy_input"),
                    ("AE sur images propres", "clean"),
                    ("AE sur images bruitées", "noisy")]:
    mse, psnr = evaluate(net, test_loader, mode)
    rows.append({"Cas": label, "MSE": round(mse, 5), "PSNR (dB)": round(psnr, 2)})
results_df = pd.DataFrame(rows)
results_df

## Étape 5 — Visualiser : originales, bruitées, reconstruites

On affiche les mêmes images sur 4 lignes : **originales**, **bruitées**, **reconstruites à partir des images propres**, et **reconstruites à partir des images bruitées** (débruitage).

In [ ]:
def show_rows(rows, titles, n=10, figsize=(14, 7)):
    fig, axes = plt.subplots(len(rows), 1, figsize=figsize)
    for ax, imgs, title in zip(axes, rows, titles):
        grid = make_grid(imgs[:n].cpu(), nrow=n, padding=2)
        ax.imshow(grid.permute(1, 2, 0).numpy())
        ax.set_title(title, fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()

images, labels = next(iter(test_loader))
images = images.to(device)
g = torch.Generator(device=device); g.manual_seed(7)
noisy = add_noise(images, NOISE_STD, g)

net.eval()
with torch.no_grad():
    rec_clean = net(images)[0]
    rec_noisy = net(noisy)[0]

show_rows([images, noisy, rec_clean, rec_noisy],
          ["Originales", f"Bruitées (σ = {NOISE_STD})",
           "AE : reconstruction des images propres", "AE : reconstruction des images bruitées (débruitage)"])

### Commentaires et comparaison de la qualité visuelle

*(Les observations ci-dessous sont les tendances attendues pour ce type de modèle ; vérifiez-les avec vos propres figures et chiffres, et adaptez si besoin.)*

1. **Reconstruction d'images propres.** Les images reconstruites sont **reconnaissables mais floues** : la forme générale, les couleurs dominantes et la position des objets sont conservées, mais les **détails fins et les textures sont perdus**. C'est normal : l'image passe par un code de 64 valeurs seulement (compression ×48) et la perte MSE favorise les moyennes lissées.
2. **Images bruitées.** Avec $\sigma = 0{,}3$, le bruit est très fort : l'image est à peine reconnaissable et le PSNR est bas (de l'ordre de 12 dB).
3. **Débruitage.** Le modèle, **sans avoir jamais vu de bruit à l'entraînement**, produit des images **beaucoup plus propres** que l'entrée bruitée : le PSNR remonte nettement (de l'ordre de 19–20 dB). La raison est le **goulot d'étranglement** : le code latent est trop petit pour contenir le bruit, qui est aléatoire donc incompressible ; seules les structures régulières de l'image sont conservées.
4. **Limite.** La sortie débruitée est un peu plus dégradée que la reconstruction d'une image propre (PSNR plus bas) : une partie de l'information réelle est perdue avec le bruit, et le résultat reste flou.
5. **Rôle du goulot.** Plus la dimension latente est grande, plus la reconstruction est fidèle, mais plus le modèle laisse aussi passer de bruit (compromis compression / qualité).

## Bonus — Denoising AE (entraîné avec du bruit)

Un **auto-encodeur débruiteur (DAE)** est entraîné à reconstruire l'image **propre** à partir d'une entrée **bruitée**. Il apprend alors explicitement à enlever le bruit, au lieu de compter uniquement sur le goulot d'étranglement.
Mettez `TRAIN_DAE = False` si vous voulez gagner du temps.

In [ ]:
TRAIN_DAE = True

if TRAIN_DAE:
    torch.manual_seed(SEED)
    dae = ConvAutoEncoder(LATENT_DIM).to(device)
    hist_dae = train(dae, train_loader, test_loader, epochs=EPOCHS, noise_std=NOISE_STD, name="DAE")

    rows = []
    for label, model, mode in [("Image bruitée (sans modèle)", net, "noisy_input"),
                               ("AE classique sur images bruitées", net, "noisy"),
                               ("Denoising AE sur images bruitées", dae, "noisy")]:
        mse, psnr = evaluate(model, test_loader, mode)
        rows.append({"Cas": label, "MSE": round(mse, 5), "PSNR (dB)": round(psnr, 2)})
    display(pd.DataFrame(rows))

    dae.eval()
    with torch.no_grad():
        rec_dae = dae(noisy)[0]
    show_rows([images, noisy, rec_noisy, rec_dae],
              ["Originales", f"Bruitées (σ = {NOISE_STD})", "AE classique (débruitage)", "Denoising AE"])

**Interprétation :** le Denoising AE obtient en général un PSNR **légèrement supérieur** à celui de l'AE classique sur les images bruitées (de l'ordre de 1 dB). L'écart est modeste car le goulot de 64 valeurs filtre déjà une grande partie du bruit ; il devient plus marqué avec un code latent plus grand ou un bruit plus structuré.

## Conclusion

- Un **auto-encodeur convolutif** (3072 → 64 → 3072) apprend une représentation compressée de CIFAR-10 sans aucune étiquette.
- Les reconstructions sont **fidèles globalement mais floues** (perte MSE + forte compression).
- L'AE **débruite** déjà les images grâce à son goulot d'étranglement ; l'entraîner explicitement avec du bruit (**Denoising AE**) améliore encore un peu le résultat.
- Pour aller plus loin : augmenter `LATENT_DIM`, ajouter `BatchNorm`, utiliser une perte SSIM/perceptuelle, ou passer à un **VAE** pour pouvoir *générer* de nouvelles images CIFAR-10.